# 01 — Análise exploratória

Este notebook estabelece **o que é possível prever** com o extrato da Locaweb antes de qualquer
modelo. Três perguntas, nesta ordem:

1. Qual é a janela de dados realmente utilizável?
2. Qual série deve ser o alvo — o volume bruto ou o volume elegível a KPI?
3. Quanto sinal existe no alvo de violação de OLA?

As respostas daqui definem decisões que atravessam todo o resto do pipeline.

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

PASTA = Path.cwd()
DADOS = PASTA / 'dados'            # saída do notebook 02, lida pelos notebooks de modelo
ARTEFATOS = PASTA / 'artefatos'    # modelos treinados por estes notebooks
DADOS.mkdir(exist_ok=True)
ARTEFATOS.mkdir(exist_ok=True)

SEMENTE = 42

## Carga dos dados

In [ ]:
# Banco SQLite da plataforma, já populado por `python manage.py importar_dataset`.
# Por padrão fica na raiz do projeto (pasta acima desta); a variável SQLITE_PATH sobrescreve.
BANCO = Path(os.environ.get('SQLITE_PATH', PASTA.parent / 'db.sqlite3'))
FUSO = 'America/Sao_Paulo'

SQL_INCIDENTES = """
SELECT i.id, i.numero, i.prioridade, i.categoria, i.subcategoria, i.descricao,
       i.aberto_em, i.resolvido_em, i.duracao_seg, i.aberto_por, i.status,
       i.entrou_kpi, i.kpi_violado, i.incidente_pai_id,
       p.codigo AS produto, e.nome AS equipe, c.codigo AS item_configuracao, f.slug AS familia_sinal
FROM core_incidente i
LEFT JOIN core_produto p          ON p.id = i.produto_id
LEFT JOIN core_equipe e           ON e.id = i.equipe_id
LEFT JOIN core_itemconfiguracao c ON c.id = i.item_configuracao_id
LEFT JOIN core_familiasinal f     ON f.id = i.familia_sinal_id
WHERE i.origem = 'dataset'
"""


def _para_local(serie):
    """O banco grava UTC; a operação (e o dataset) vive em horário de São Paulo."""
    return pd.to_datetime(serie, format='mixed', utc=True).dt.tz_convert(FUSO).dt.tz_localize(None)


def carregar_incidentes(banco=BANCO):
    if not Path(banco).exists():
        raise FileNotFoundError(f'Banco não encontrado em {banco}. Rode `python manage.py importar_dataset`.')
    with sqlite3.connect(f'file:{banco}?mode=ro', uri=True) as conexao:
        df = pd.read_sql_query(SQL_INCIDENTES, conexao)

    df['aberto_em'] = _para_local(df['aberto_em'])
    df['resolvido_em'] = _para_local(df['resolvido_em'])
    df['data'] = df['aberto_em'].dt.normalize()
    df['entrou_kpi'] = df['entrou_kpi'].astype(bool)
    # Nulo em quem não entrou no KPI: "não violou" é diferente de "nem foi medido".
    df['kpi_violado'] = df['kpi_violado'].astype('boolean')
    df['mttr_min'] = (df['resolvido_em'] - df['aberto_em']).dt.total_seconds() / 60
    df.loc[~df['entrou_kpi'], 'mttr_min'] = pd.NA
    for coluna in ('produto', 'equipe', 'item_configuracao', 'familia_sinal', 'categoria'):
        df[coluna] = df[coluna].fillna('').astype(str)
    return df


def detectar_inicio_regime(serie_diaria, janela=28, fracao=0.25):
    """Primeiro dia a partir do qual a mediana móvel de 28 dias nunca mais cai abaixo de 25% da
    mediana do período final. Separa operação real de carga histórica parcial (backfill)."""
    serie = pd.Series(serie_diaria).sort_index()
    referencia = serie.tail(janela * 3).median()
    if not referencia:
        return serie.index.min()
    movel = serie.rolling(janela, min_periods=janela).median()
    abaixo = movel[movel < referencia * fracao]
    if abaixo.empty:
        return serie.index.min()
    posicao = serie.index.get_loc(abaixo.index.max())
    return serie.index[min(posicao + 1, len(serie) - 1)]

In [ ]:
incidentes = carregar_incidentes()
resumo = {
    'incidentes': len(incidentes),
    'elegiveis_kpi': int(incidentes['entrou_kpi'].sum()),
    'violacoes': int((incidentes['kpi_violado'] == True).sum()),  # noqa: E712
    'produtos': incidentes['produto'].replace('', pd.NA).nunique(),
    'equipes': incidentes['equipe'].replace('', pd.NA).nunique(),
    'itens_configuracao': incidentes['item_configuracao'].replace('', pd.NA).nunique(),
    'familias': incidentes['familia_sinal'].replace('', pd.NA).nunique(),
    'inicio': incidentes['data'].min().date(),
    'fim': incidentes['data'].max().date(),
}
for chave, valor in resumo.items():
    print(f'{chave:>20}: {valor}')

## 1. A janela utilizável

O extrato cobre três anos, mas a densidade não é uniforme. A tabela mensal mostra que 2023 e 2024
somam menos incidentes elegíveis do que um único mês de 2025.

In [ ]:
diario = incidentes.groupby('data').agg(total=('numero', 'size'), kpi=('entrou_kpi', 'sum'))
diario['violacoes'] = (
    incidentes.assign(v=incidentes['kpi_violado'].fillna(False).astype(bool).astype(int))
    .groupby('data')['v'].sum()
)
mensal = diario.resample('MS').sum()
mensal.tail(18)

In [ ]:
fig, eixo = plt.subplots()
eixo.plot(diario.index, diario['total'], lw=0.8, label='volume bruto', color='#94a3b8')
eixo.plot(diario.index, diario['kpi'], lw=1.0, label='elegível a KPI', color='#2563eb')
eixo.set_title('Volume diário de incidentes — bruto vs. elegível a KPI')
eixo.legend()
plt.show()

### A ruptura de setembro/2025

O volume bruto salta a partir de 01/09/2025, enquanto o volume **elegível a KPI** segue na mesma
faixa. A comparação abaixo mede essa diferença.

**Decisão que sai daqui:** o alvo dos modelos de volume é `total_kpi` (incidentes elegíveis a KPI),
não o volume bruto.

In [ ]:
corte = pd.Timestamp('2025-09-01')
antes = diario.loc['2025-01-01':corte - pd.Timedelta(days=1)]
depois = diario.loc[corte:]
comparacao = pd.DataFrame({
    'antes de 01/09': [antes['total'].mean(), antes['kpi'].mean()],
    'a partir de 01/09': [depois['total'].mean(), depois['kpi'].mean()],
}, index=['volume bruto/dia', 'elegível a KPI/dia']).round(1)
comparacao['variação %'] = ((comparacao['a partir de 01/09'] / comparacao['antes de 01/09'] - 1) * 100).round(1)
comparacao

In [ ]:
# O que caracteriza o volume que ficou fora do KPI: origem da abertura, status e prioridade.
for coluna in ('aberto_por', 'status', 'prioridade'):
    tabela = pd.crosstab(incidentes[coluna], incidentes['entrou_kpi'], normalize='columns').mul(100).round(1)
    tabela.columns = ['fora do KPI (%)', 'elegível a KPI (%)']
    display(tabela)

### Detecção do início do regime

O corte não é escolhido a dedo: `detectar_inicio_regime` procura o primeiro dia a partir do qual a
mediana móvel de 28 dias nunca mais cai abaixo de 25% da mediana do período final.

In [ ]:
serie_kpi = diario['kpi'].reindex(pd.date_range(diario.index.min(), diario.index.max()), fill_value=0)
inicio = detectar_inicio_regime(serie_kpi)
print('Início do regime operacional detectado:', inicio.date())
print('Dias utilizáveis:', (serie_kpi.index.max() - inicio).days + 1)
print('Incidentes descartados (backfill):', int((incidentes['data'] < inicio).sum()))

## 2. Sazonalidade do alvo

O perfil semanal é o que qualquer modelo precisa capturar — e é por isso que a baseline de
comparação é a sazonal ingênua (repetir o mesmo dia da semana anterior), e não a média simples.

In [ ]:
regime = diario.loc[inicio:]
perfil = regime.groupby(regime.index.dayofweek)['kpi'].agg(['mean', 'std', 'count'])
perfil.index = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom'][:len(perfil)]
perfil['índice'] = (perfil['mean'] / regime['kpi'].mean()).round(2)
perfil.round(1)

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.6))
perfil['mean'].plot(kind='bar', ax=a, color='#2563eb')
a.set_title('Média de incidentes elegíveis por dia da semana')
regime['kpi'].rolling(7).mean().plot(ax=b, color='#2563eb')
b.set_title('Média móvel de 7 dias — nível da operação')
plt.tight_layout(); plt.show()

## 3. Quanto sinal existe para prever violação de OLA

Violação de OLA é um evento raro — a limitação mais séria do dataset, explícita antes de qualquer modelo.

In [ ]:
elegiveis = incidentes[incidentes['entrou_kpi'] & incidentes['kpi_violado'].notna()].copy()
elegiveis = elegiveis[elegiveis['data'] >= inicio]
elegiveis['violou'] = elegiveis['kpi_violado'].astype(bool).astype(int)

print(f'Incidentes elegíveis no regime : {len(elegiveis):,}')
print(f'Violações                      : {elegiveis["violou"].sum():,}')
print(f'Taxa de violação               : {elegiveis["violou"].mean() * 100:.2f}%')
print(f'Dias com ao menos uma violação : {(regime["violacoes"] > 0).sum()} de {len(regime)}')

In [ ]:
por_prioridade = elegiveis.groupby('prioridade')['violou'].agg(['size', 'sum', 'mean'])
por_prioridade.columns = ['incidentes', 'violações', 'taxa']
display(por_prioridade.round(4))

por_equipe = elegiveis.groupby('equipe')['violou'].agg(['size', 'sum', 'mean'])
por_equipe.columns = ['incidentes', 'violações', 'taxa']
por_equipe = por_equipe[por_equipe['incidentes'] >= 200].sort_values('taxa', ascending=False)
print('Taxa por equipe (≥200 incidentes) — a variação entre equipes é o que dá sinal ao modelo:')
display(por_equipe.round(4))

In [ ]:
por_origem = elegiveis.groupby('aberto_por')['violou'].agg(['size', 'mean']).round(4)
por_origem.columns = ['incidentes', 'taxa de violação']
por_origem

**Conclusões que viram decisão de modelagem:**

- Classe severamente desbalanceada ⇒ a métrica principal do modelo de OLA é **PR-AUC**, não
  acurácia, e o limiar de decisão é escolhido na validação.
- A taxa varia entre equipes e entre chamados manuais e automáticos: existe sinal a aprender.
- Treinar com peso de classe distorce a escala da probabilidade ⇒ a saída precisa ser **calibrada**.